# Data Inspection using DuckDB

Continuing with Data Integrity Check using DuckDB due to better cpu and memory efficiency.



In [2]:
import duckdb
import pandas as pd

DB = "../data/tfl-cycling-journey-data-2024-2025/tfl_cycle_hires.duckdb"
con = duckdb.connect(DB)
con.sql("SHOW TABLES")

┌───────────────────┐
│       name        │
│      varchar      │
├───────────────────┤
│ cycle_hires_table │
└───────────────────┘

In [3]:
con.sql("DESCRIBE cycle_hires_table")

┌──────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │ column_type │  null   │   key   │ default │  extra  │
│       varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ Number               │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Start date           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Start station number │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Start station        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ End date             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ End station number   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ End station          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Bike number          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Bike model           │ VARCHAR     │ YES     │ NUL

- Data types need correcting (Start date VARCHAR to TIMESTAMP)

# Initial Inspection

In [4]:
#Does anything look odd or out of place?
df = con.sql('''SELECT * 
                FROM cycle_hires_table
                WHERE Number IN (138806120,138807817,140335775,143181920) -- Known ID's with issues;
                ''').df()

display(df)

,Number,Start date,Start station number,Start station,End date,End station number,End station,Bike number,Bike model,Total duration,Total duration (ms)
0,138807817,2024-04-24 14:26,200156,"Pott Street, Bethnal Green",2024-04-24 14:34,200157,"Fournier Street, Whitechapel",14403,CLASSIC,7m 41s,461384
1,138806120,2024-04-24 12:48,001037,"Park Lane , Hyde Park",2024-04-24 14:33,001037,"Park Lane , Hyde Park",30293,CLASSIC,1h 45m 11s,6311784
2,140335775,2024-06-19 16:46,002681,"Harriet Street, Knightsbridge",2024-06-19 17:06,200211,"Union Grove, Wandsworth Road",62223,PBSC_EBIKE,20m 38s,1238811
3,143181920,24/09/2024 19:00,3463,"Argyll Road, Kensington",24/09/2024 19:16,300030,"Thorndike Close, West Chelsea",40290,CLASSIC,15m 46s,946540


## Problems
 - Date format, under Start date and End date columns, is not always consistent. For example, sometimes date is formatted as [dd/mm/yyyy hh:mm] other times it's [yyyy-mm-dd hh:mm]
   - Will need to reformat the date and then convert it into a datetime variable (TIMESTAMP) 
 - Start station number & End station number have some values that lead with 0 and other that do not - Do these represent the same station number ID?
   - This is corrected later in this notebook 
 - We need to see if Station Number ID's match with Station names - Are they always consistent.
 - Does the Total duration and Total duration (ms) match up properly and equal? If so, we can just use the Total duration (ms) for data analysis
 - Need to see if Bike Number ID has any missing values
   - there should be duplicates in this column because the same bike would most definitely be used on multiple occassions over a 2 year period.
 - Later, I'll need to merge a new table containing station names with geographical coordinates (longitude, latitude) and weather data.


In [5]:
# Total Number of Rows in cycle_hires_table

df = con.sql('''SELECT COUNT(*) AS "Total No. of Records" FROM cycle_hires_table''').df()
display(df)
total_records = df['Total No. of Records'].item() 

,Total No. of Records
0,17823393


# Performing Data Quality Checks

## Check For Missing Data
- Are there any records that do not provide data for all 11 variables? 
- We'll first check for blank cells and NULL values, then for any other values that represent unknowns lile 'na or 'n/a' etc.

In [7]:
df = con.sql('''SELECT *
FROM cycle_hires_table
WHERE
    Number IS NULL OR
    "Start date" IS NULL OR "Start date" = '' OR
    "Start station number" IS NULL OR "Start station number" = '' OR
    "Start station" IS NULL OR "Start station" = '' OR
    "End date" IS NULL OR "End date" = '' OR
    "End station number" IS NULL OR "End station number" = '' OR
    "End station" IS NULL OR "End station" = '' OR
    "Bike number" IS NULL OR "Bike number" = '' OR
    "Bike model" IS NULL OR "Bike model" = '' OR
    "Total duration" IS NULL OR "Total duration" = '' OR
    "Total duration (ms)" IS NULL
''').df()
display(df)
print(f"There are {df.shape[0]} records that have at least one piece of missing data.\n\n")


,Number,Start date,Start station number,Start station,End date,End station number,End station,Bike number,Bike model,Total duration,Total duration (ms)
0,140976015,2024-07-11 11:19,001225,"George Street, Marylebone",2024-07-11 11:58,001071,"Tower Gardens , Tower",NaN,CLASSIC,39m 25s,2365940
1,149460374,2025-06-15 15:15,002680,"Harrowby Street, Marylebone",NaN,NaN,NaN,51139,CLASSIC,NaN,<NA>
2,149455940,2025-06-15 13:01,300038,"Star Road, West Kensington",NaN,NaN,NaN,21293,CLASSIC,NaN,<NA>
3,149448920,2025-06-15 06:49,300001,"Sandilands Road, Walham Green",NaN,NaN,NaN,57676,CLASSIC,NaN,<NA>
4,149445836,2025-06-14 21:46,001182,"Notting Hill Gate Station, Notting Hill",NaN,NaN,NaN,55089,CLASSIC,NaN,<NA>
...,...,...,...,...,...,...,...,...,...,...,...
180,151343530,2025-08-17 16:00,300041,"Finnis Street, Bethnal Green",NaN,NaN,NaN,61110,PBSC_EBIKE,NaN,<NA>
181,151338530,2025-08-17 14:12,001075,"Hyde Park Corner, Hyde Park",NaN,NaN,NaN,63348,PBSC_EBIKE,NaN,<NA>
182,151338324,2025-08-17 14:08,300209,"Albert Square, Stockwell",NaN,NaN,NaN,60329,PBSC_EBIKE,NaN,<NA>
183,151316957,2025-08-16 16:53,001137,"Harper Road, The Borough",NaN,NaN,NaN,23912,CLASSIC,NaN,<NA>


There are 185 records that have at least one piece of missing data.




- One possible reason for missing data for End station but not start station could be that bikes where never returned or bikes were decommissioned or taken for maintenance. 

In [8]:
#Which columns have missing data and how much
print("Columns that have missing data:")
df = con.sql('''SELECT
    COUNT(*) FILTER (WHERE Number IS NULL) AS number_missing,
    COUNT(*) FILTER (WHERE NULLIF("Start date", '') IS NULL) AS start_date_missing,
    COUNT(*) FILTER (WHERE NULLIF("Start station number", '') IS NULL) AS start_station_number_missing,
    COUNT(*) FILTER (WHERE NULLIF("Start station", '') IS NULL) AS start_station_missing,
    COUNT(*) FILTER (WHERE NULLIF("End date", '') IS NULL) AS end_date_missing,
    COUNT(*) FILTER (WHERE NULLIF("End station number", '') IS NULL) AS end_station_number_missing,
    COUNT(*) FILTER (WHERE NULLIF("End station", '') IS NULL) AS end_station_missing,
    COUNT(*) FILTER (WHERE NULLIF("Bike number", '') IS NULL) AS bike_number_missing,
    COUNT(*) FILTER (WHERE NULLIF("Bike model", '') IS NULL) AS bike_model_missing,
    COUNT(*) FILTER (WHERE NULLIF("Total duration", '') IS NULL) AS total_duration_missing,
    COUNT(*) FILTER (WHERE "Total duration (ms)" IS NULL) AS total_duration_ms_missing
FROM cycle_hires_table;
''').df()
display(df)


Columns that have missing data:


,number_missing,start_date_missing,start_station_number_missing,start_station_missing,end_date_missing,end_station_number_missing,end_station_missing,bike_number_missing,bike_model_missing,total_duration_missing,total_duration_ms_missing
0,0,0,0,0,184,184,184,1,0,184,184


## Checking for Duplicate Rows

In [9]:
#Making sure that there are no duplicate entries in the database table
df = con.sql('''
           SELECT * 
           FROM cycle_hires_table
           GROUP BY ALL
           HAVING COUNT(*) > 1
        '''
       ).df()
if df.shape[0] == 0:
    print("There are no duplicate entries for this table \n")
else:
    print("Duplicate entries found! See table below: \n")

display(df)

There are no duplicate entries for this table 



,Number,Start date,Start station number,Start station,End date,End station number,End station,Bike number,Bike model,Total duration,Total duration (ms)


## Column by Column Checks

 - We checked the database for NULL and blank cells but we haven't checked for other types of text that can represent missing or unknown values such as N/a or n/a or 999 or 'unknown', etc.
 - We can do this for one column at a time - for example, by listing out all the discrete values in 'Start station' etc.


### Checking: Number

In [10]:
#Number column
df = con.sql("SUMMARIZE SELECT Number FROM cycle_hires_table").df()
display(df)
print(f"The minimum value in the Number columns is: {df['min'][0]}, and maximum number is: {df['max'][0]}")

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,Number,BIGINT,136450332,154763894,17648642,145604763.30466297,5286946.495078557,141004684,145628246,150188986,17823393,0.0


The minimum value in the Number columns is: 136450332, and maximum number is: 154763894


- Number column has no missing values and data type is BIGINT

#### Duplicates in Number Column?

In [11]:
#Are there any duplicates in just the "Number" column? 
df = con.sql('''SELECT 
                  Number, COUNT(*) AS Occurrence
                FROM 
                  cycle_hires_table
                GROUP BY 
                  Number
                HAVING 
                  COUNT(*) > 1
''').df()
if df.shape[0] == 0:
    print("There are no duplicate Numbers for this table \n")
else:
    print("Duplicate entries found! See table below: \n")
display(df)


There are no duplicate Numbers for this table 



,Number,Occurrence


### Checking: Start Date

In [12]:
#"Start date" Column
#NOTE: This column has no missing values as shown above.
#We'll look at the inconsistent date formats as well as seeing whether there are 'Other' values mixed in

df = con.sql('''
SELECT
   CASE
        WHEN "Start date" IS NULL OR "Start date" = '' THEN 'Missing/Empty'
        WHEN TRY_STRPTIME("Start date", '%Y-%m-%d %H:%M') IS NOT NULL THEN 'YYYY-MM-DD'
        WHEN TRY_STRPTIME("Start date", '%m/%d/%Y %H:%M') IS NOT NULL THEN 'MM/DD/YYYY'
        WHEN TRY_STRPTIME("Start date", '%d/%m/%Y %H:%M') IS NOT NULL THEN 'DD/MM/YYYY'
        WHEN TRY_STRPTIME("Start date", '%d-%b-%Y %H:%M') IS NOT NULL THEN 'DD-Mon-YYYY'
        WHEN TRY_STRPTIME("Start date", '%b %d, %Y %H:%M') IS NOT NULL THEN 'Mon DD, YYYY'
        WHEN TRY_STRPTIME("Start date", '%Y%m%d %H:%M') IS NOT NULL THEN 'YYYYMMDD'
     ELSE 'Other'
   END AS Date_Format,
   COUNT(*) AS count
FROM cycle_hires_table
GROUP BY Date_Format
ORDER BY count DESC
''').df()
display(df)


,Date_Format,count
0,YYYY-MM-DD,16162223
1,DD/MM/YYYY,996104
2,MM/DD/YYYY,665066


- Vast majority of Start dates are in the format of YYYY-MM-DD. Will need to convert all dates in this column into this format.

### Checking: End Date

In [13]:
#"End date" Column
#NOTE: This column HAS missing values as shown above (Columns that have missing data:)
#We'll look at the inconsistent date formats as well as seeing whether there are 'Other' values mixed in

df = con.sql('''
SELECT
   CASE
        WHEN "End date" IS NULL OR "End date" = '' THEN 'Missing/Empty'
        WHEN TRY_STRPTIME("End date", '%Y-%m-%d %H:%M') IS NOT NULL THEN 'YYYY-MM-DD'
        WHEN TRY_STRPTIME("End date", '%m/%d/%Y %H:%M') IS NOT NULL THEN 'MM/DD/YYYY'
        WHEN TRY_STRPTIME("End date", '%d/%m/%Y %H:%M') IS NOT NULL THEN 'DD/MM/YYYY'
        WHEN TRY_STRPTIME("End date", '%d-%b-%Y %H:%M') IS NOT NULL THEN 'DD-Mon-YYYY'
        WHEN TRY_STRPTIME("End date", '%b %d, %Y %H:%M') IS NOT NULL THEN 'Mon DD, YYYY'
        WHEN TRY_STRPTIME("End date", '%Y%m%d %H:%M') IS NOT NULL THEN 'YYYYMMDD'
        WHEN TRY_STRPTIME("End date", '%Y%m%d %H:%M') IS NOT NULL THEN 'YYYYMMDD'
     ELSE 'Other'
   END AS Date_Format,
   COUNT(*) AS count
FROM cycle_hires_table
GROUP BY Date_Format
ORDER BY count DESC
''').df()
display(df)

,Date_Format,count
0,YYYY-MM-DD,16162039
1,DD/MM/YYYY,996265
2,MM/DD/YYYY,664905
3,Missing/Empty,184


- Will need to convert these two columns (Start date, End date) into TIMESTAMP with a date format of YYYY-MM-DD HH:MM

### Checking: Start station number

In [14]:
# Let's first make sure "Start station number" Column doesn't contain any value other than 0-9 (we don't want other characters in this column)

df = con.sql("""SELECT COUNT(*) AS count
FROM cycle_hires_table
WHERE REGEXP_MATCHES("Start station number", '[^0-9]')
""").df()
print(f"There are: {df['count'][0]} values containing characters that are NOT a digit between 0 and 9")
display(df)     
if (df['count'][0] == 0):
    print("Great! This column only contains numbers in VARCHAR format.")

There are: 0 values containing characters that are NOT a digit between 0 and 9


,count
0,0


Great! This column only contains numbers in VARCHAR format.


### Checking: Start station number AND Start station

In [15]:
#Are there multiple station numbers per station?
#There should be only one "Start station number" per "Start station"
df = con.sql('''
SELECT DISTINCT "Start station", "Start station number"
FROM cycle_hires_table
WHERE "Start station" IN
    (SELECT "Start station"
    FROM cycle_hires_table
    GROUP BY "Start station"
    HAVING COUNT(DISTINCT "Start station number") > 1)
ORDER BY "Start station", "Start station number"
''').df()
display(df)
if (len(df['Start station'])>0):
    print(f"We have multiple start station numbers per Station name.")

,Start station,Start station number
0,"Abbey Orchard Street, Westminster",003429
1,"Abbey Orchard Street, Westminster",3429
2,"Abbey Orchard Street, Westminster_OLD",003429444
3,"Abbey Orchard Street, Westminster_OLD",3429444
4,"Aberdeen Place, St. John's Wood",002698
...,...,...
843,"Worship Street, Shoreditch",997
844,"Wren Street, Holborn",022169
845,"Wren Street, Holborn",22169
846,"Wright's Lane, Kensington",001094


We have multiple start station numbers per Station name.


- We have a problem with <b>leading zeros</b> in "Start station number" column
- We would need to strip any leading zeros in this field and see if there are any duplicates left. Will do this, again, when Altering the table.

In [12]:
#Let's check if we have any duplicates left when we remove the leading zeros. 
df['Start station number'] = pd.to_numeric(df['Start station number'], errors='coerce').astype('Int64')
print("Leading zeros have been removed from Start station number Column")
display(df)

#Because leading zeros have been removed, there shouldn't be any duplicates remaining
df_unique = df.drop_duplicates(keep=False)
print("The duplicates remaining:")
display(df_unique)


Leading zeros have been removed from Start station number Column


,Start station,Start station number
0,"Abbey Orchard Street, Westminster",3429
1,"Abbey Orchard Street, Westminster",3429
2,"Abbey Orchard Street, Westminster_OLD",3429444
3,"Abbey Orchard Street, Westminster_OLD",3429444
4,"Aberdeen Place, St. John's Wood",2698
...,...,...
843,"Worship Street, Shoreditch",997
844,"Wren Street, Holborn",22169
845,"Wren Street, Holborn",22169
846,"Wright's Lane, Kensington",1094


The duplicates remaining:


,Start station,Start station number


- Duplicates caused by leading zeros only in Start station number column. This means that we can remove duplcates simply by converting this column into an Int in DuckDB

### Checking: End station number

In [16]:
#With End Station Number, are there any fields with non-numbered characters?
#This will flag if any value is anything other than 0-9
df = con.sql("""SELECT COUNT(*) AS count
                FROM cycle_hires_table
                WHERE REGEXP_MATCHES("End station number", '[^0-9]')
            """).df()
print(f"There are: {df['count'][0]} values containing characters that are NOT a digit between 0 and 9")
display(df)

There are: 0 values containing characters that are NOT a digit between 0 and 9


,count
0,0


### Checking: End station number AND End station

In [18]:
#Multiple End station numbers per station name?
#We'll probably see leading zeroes again in End station number
df = con.sql("""
SELECT DISTINCT "End Station", "End station number"
FROM cycle_hires_table
WHERE "End station" IN (
  SELECT "End station"
  FROM cycle_hires_table
  GROUP BY "End Station"
  HAVING COUNT(DISTINCT "End station number")>1)
ORDER BY "End Station", "End station number";
""").df()
display(df)

,End station,End station number
0,"Abbey Orchard Street, Westminster",003429
1,"Abbey Orchard Street, Westminster",3429
2,"Abbey Orchard Street, Westminster_OLD",003429444
3,"Abbey Orchard Street, Westminster_OLD",3429444
4,"Aberdeen Place, St. John's Wood",002698
...,...,...
847,"Worship Street, Shoreditch",997
848,"Wren Street, Holborn",022169
849,"Wren Street, Holborn",22169
850,"Wright's Lane, Kensington",001094


- For Start station number and End station number, we can easily remove leading zeros by altering the columns from VARCHAR to INT
- All permanent changes to the database table will be done in the next notebook 

In [19]:
#Again, let's check if we have any duplicates left when we remove the leading zeros. 
df['End station number'] = pd.to_numeric(df['End station number'], errors='coerce').astype('Int64')
print("Leading zeros have been removed from End station number Column")
display(df)

#Because leading zeros have been removed, there shouldn't be any duplicates remaining!
df_unique = df.drop_duplicates(keep=False)
print("The duplicates remaining:")
display(df_unique)

Leading zeros have been removed from End station number Column


,End station,End station number
0,"Abbey Orchard Street, Westminster",3429
1,"Abbey Orchard Street, Westminster",3429
2,"Abbey Orchard Street, Westminster_OLD",3429444
3,"Abbey Orchard Street, Westminster_OLD",3429444
4,"Aberdeen Place, St. John's Wood",2698
...,...,...
847,"Worship Street, Shoreditch",997
848,"Wren Street, Holborn",22169
849,"Wren Street, Holborn",22169
850,"Wright's Lane, Kensington",1094


The duplicates remaining:


,End station,End station number


- No dupliocates are remaining after the leading zeros have been removed which means we can simply convert this colomn into an INT in the database table to remove leading zeros and therefore duplicates.

### Checking: "Start station" and "Start station number" Match with "End station" and "End station number"

- In the next couple of cells, will be merging columns together (i.e. Start station, Start station number) and using sets to check for symmetric difference.

In [23]:
#For example, if "997" represents "Worship Street, Shoreditch" for the End Station, is this true for Start station?

#creatying a dataframe for starting location
#This will be merged and compared with End station
start_stations = con.sql("""SELECT DISTINCT
"Start station", CAST("Start station number" AS INT) AS "Start station number" -- Casting as INT simply to remove duplicates
FROM cycle_hires_table
ORDER BY "Start station";""").df()

display(start_stations)



,Start station,Start station number
0,"Abbey Orchard Street, Westminster",3429
1,"Abbey Orchard Street, Westminster_OLD",3429444
2,"Abbotsbury Road, Holland Park",200111
3,"Aberdeen Place, St. John's Wood",2698
4,"Aberfeldy Street, Poplar",200078
...,...,...
830,"Wren Street, Holborn",22169
831,"Wright's Lane, Kensington",1094
832,"Wynne Road, Stockwell",300230
833,"York Hall, Bethnal Green",200042


In [24]:
#Dataframe for end station data
#Can now convert both these dataframse into sets and do a comparison
end_stations = con.sql("""SELECT DISTINCT 
                          "End station",
                          CAST("End station number" AS INT) AS "End station number"
                          FROM cycle_hires_table
                          ORDER BY "End station";
                      """).df()
display(end_stations)

,End station,End station number
0,"Abbey Orchard Street, Westminster",3429
1,"Abbey Orchard Street, Westminster_OLD",3429444
2,"Abbotsbury Road, Holland Park",200111
3,"Aberdeen Place, St. John's Wood",2698
4,"Aberfeldy Street, Poplar",200078
...,...,...
833,"Wright's Lane, Kensington",1094
834,"Wynne Road, Stockwell",300230
835,"York Hall, Bethnal Green",200042
836,"York Way, Kings Cross",300235


Why are there more End stations than start stations? Possibly because New stations were opened where riders ended their journeys but no one had yet started a journey from these locations.

In [25]:
#Concatinating columns to make comparision between start_stations and end_stations easier
start_combined = pd.DataFrame({
    "Combined": start_stations["Start station"] + start_stations["Start station number"].astype(str) })

display(start_combined)

,Combined
0,"Abbey Orchard Street, Westminster3429"
1,"Abbey Orchard Street, Westminster_OLD3429444"
2,"Abbotsbury Road, Holland Park200111"
3,"Aberdeen Place, St. John's Wood2698"
4,"Aberfeldy Street, Poplar200078"
...,...
830,"Wren Street, Holborn22169"
831,"Wright's Lane, Kensington1094"
832,"Wynne Road, Stockwell300230"
833,"York Hall, Bethnal Green200042"


In [26]:
end_combined = pd.DataFrame({
    "Combined": end_stations["End station"] + end_stations["End station number"].astype(str)})

display(end_combined)

,Combined
0,"Abbey Orchard Street, Westminster3429"
1,"Abbey Orchard Street, Westminster_OLD3429444"
2,"Abbotsbury Road, Holland Park200111"
3,"Aberdeen Place, St. John's Wood2698"
4,"Aberfeldy Street, Poplar200078"
...,...
833,"Wright's Lane, Kensington1094"
834,"Wynne Road, Stockwell300230"
835,"York Hall, Bethnal Green200042"
836,"York Way, Kings Cross300235"


In [27]:
#finding if there are any differences between both station columns
#If there is a difference, this might mean there have been mispellings in station names or incorrect mapping between names and numbers

start_set = set(start_combined["Combined"])
end_set = set(end_combined["Combined"])

#items in start but not in end
only_in_start = start_set - end_set

#items only in end but not in start
only_in_end = end_set - start_set

#Items in either one but not in both sets (symmetric difference)
all_diff = start_set ^ end_set


In [28]:

print("Stations in start_set but not in end_set: ",only_in_start or "The start_set is empty")

print("Stations in end_set but not in start_set: ",only_in_end or "The end_set is empty")

print("All unique differences: ", all_diff or "There are no symmetric difference")



Stations in start_set but not in end_set:  The start_set is empty
Stations in end_set but not in start_set:  {'London Fields, Hackney200025', nan, 'Mechanical Workshop Penton10626'}
All unique differences:  {'Mechanical Workshop Penton10626', 'London Fields, Hackney200025', nan}


 - There doesn't seem to be any mispellings in station names or incorrect mapping between names and numbers.
 - No one has hired a bike from either 'Mechanical Workshop Penton' or 'London Fields, Hackney' but they have ended their journey there
 - Let's investigate these End Stations further.

In [29]:
#How many journeys ended in London Fields, Hackney?
con.sql("""SELECT "Start date", "Start station", CAST("Start station number" AS INT) AS "Start station number",
                  "End date", "End station", CAST("End station number" AS INT) AS "End station number", 
                  "Total duration", "Total duration (ms)"
           FROM cycle_hires_table
           WHERE "End station" = 'London Fields, Hackney'
           ORDER BY "Total duration (ms)" DESC;""")

┌──────────────────┬──────────────────────────────┬──────────────────────┬──────────────────┬────────────────────────┬────────────────────┬─────────────────┬─────────────────────┐
│    Start date    │        Start station         │ Start station number │     End date     │      End station       │ End station number │ Total duration  │ Total duration (ms) │
│     varchar      │           varchar            │        int32         │     varchar      │        varchar         │       int32        │     varchar     │        int64        │
├──────────────────┼──────────────────────────────┼──────────────────────┼──────────────────┼────────────────────────┼────────────────────┼─────────────────┼─────────────────────┤
│ 2025-06-17 18:49 │ Stratford Station, Stratford │               300234 │ 2025-09-22 17:10 │ London Fields, Hackney │             200025 │ 96d 22h 21m 37s │          8374897984 │
└──────────────────┴──────────────────────────────┴──────────────────────┴──────────────────┴───────

- Only One journey was made to London Fields, Hackney in 2 years worth of cycle hires, lasting 96 days. 
- And, there's been zero hires from London Fields, Hackney in the same 2 years!
- This could be because London Fields, Hackney only existed for a very short time or was temporary for bike testing purposes by staff. 

In [30]:
#How many journeys ended in 'Mechanical Workshop Penton'?
con.sql("""SELECT "Start date", "Start station", CAST("Start station number" AS INT) AS "Start station number",
                  "End date", "End station", CAST("End station number" AS INT) AS "End station number", 
                  "Total duration", "Total duration (ms)"
           FROM cycle_hires_table
           WHERE "End station" = 'Mechanical Workshop Penton'
           ORDER BY "Total duration (ms)" DESC;""")

┌──────────────────┬───────────────────────────────────────────────┬──────────────────────┬──────────────────┬────────────────────────────┬────────────────────┬──────────────────┬─────────────────────┐
│    Start date    │                 Start station                 │ Start station number │     End date     │        End station         │ End station number │  Total duration  │ Total duration (ms) │
│     varchar      │                    varchar                    │        int32         │     varchar      │          varchar           │       int32        │     varchar      │        int64        │
├──────────────────┼───────────────────────────────────────────────┼──────────────────────┼──────────────────┼────────────────────────────┼────────────────────┼──────────────────┼─────────────────────┤
│ 2025-06-12 15:52 │ Portugal Street, Holborn                      │                 3456 │ 2025-12-10 14:42 │ Mechanical Workshop Penton │              10626 │ 180d 23h 50m 46s │         1563

- Mechanical Workshop Penton is most likely bikes sent in for Maintenance & Repair
- So we could imagine bikes are inspected at the start station and then, if faulty or old, are taken to Mechanical Workshop Penton and simply logged as a cycle hire (maybe by mistake).
- Also notice that out of nearly 18million cycle hires over two years, only 723 bicycles have been recorded as being taken to Mechanical Workshop Penton. Why so few?

### Checking: Bike number

In [24]:
#We have 1 missing value in the Bike number column
#Are there any values that are not numbers in the Bike number column?
con.sql("""SELECT COUNT(*) AS count
FROM cycle_hires_table
WHERE REGEXP_MATCHES("Bike number", '[^0-9]') > 0""")

┌───────┐
│ count │
│ int64 │
├───────┤
│     0 │
└───────┘

In [25]:
#Counting the number of unique bike numbers over this 2 year dataset
#It's not known, at this point, whether bikes that are replaced get new bike id numbers or reuse old bike id numbers
con.sql("""SELECT COUNT(DISTINCT "Bike number") AS "Total Unique Bikes"
FROM cycle_hires_table
""")

┌────────────────────┐
│ Total Unique Bikes │
│       int64        │
├────────────────────┤
│              15743 │
└────────────────────┘

### Checking: Bike model

In [26]:
#What types of Bike Model do we have?
df = con.sql("""SELECT DISTINCT "Bike Model", COUNT(*) AS "Model Count"
           FROM cycle_hires_table
           GROUP BY "Bike model"
           ORDER BY "Model Count" DESC
           """).df()
df['Ratio'] = round(df['Model Count']/df['Model Count'].sum(), 2)
display(df)

,Bike model,Model Count,Ratio
0,CLASSIC,15188787,0.85
1,PBSC_EBIKE,2634606,0.15


- Only 15% of bicycles are E-bikes (use an electric motor to assist riders as they pedal)
- More info here: https://tfl.gov.uk/info-for/media/press-releases/2022/august/santander-cycles-to-launch-e-bikes-in-london-from-september
- Are e-bike use higher in 2025 than in 2024?
- Are e-bikes more popular in certain conditions?

### Checking: "Total duration" AND "Total duration (ms)"

- Do the durations match up between both columns or is ther anything odd like: "Total duration" = 29d 14h 2m 9s AND "Total duration (ms)" = 172

In [33]:
#Quick check
con.sql("""
            SELECT "Total duration","Total duration (ms)"  FROM cycle_hires_table ORDER By Random()
            ;""")

┌────────────────┬─────────────────────┐
│ Total duration │ Total duration (ms) │
│    varchar     │        int64        │
├────────────────┼─────────────────────┤
│ 16m 16s        │              976943 │
│ 10m 2s         │              602363 │
│ 4m 23s         │              263499 │
│ 11m 37s        │              697633 │
│ 17m 58s        │             1078290 │
│ 49m 46s        │             2986663 │
│ 21m 2s         │             1262250 │
│ 19m 8s         │             1148868 │
│ 9m 36s         │              576470 │
│ 12m 45s        │              765375 │
│   ·            │                 ·   │
│   ·            │                 ·   │
│   ·            │                 ·   │
│ 6m 57s         │              417963 │
│ 16m 27s        │              987455 │
│ 11m 40s        │              700900 │
│ 13m 47s        │              827352 │
│ 39m 33s        │             2373565 │
│ 7m 22s         │              442470 │
│ 4m 50s         │              290250 │
│ 41m 8s        

In [34]:
#Testing that the time duration in 'Total duration' matches with the duration in 'Total duration (ms)'
#If it does, we should end up with an empty dataframe at the bottom
df = con.sql(
    """
    WITH new_times AS(
      SELECT "Total duration", "Total duration (ms)",
      (CAST("Total duration" AS INTERVAL).epoch_ms() // 1000 ) AS string_sec,
      ("Total duration (ms)" // 1000) AS column_sec
      FROM cycle_hires_table
    )
    SELECT
      "Total duration", "Total duration (ms)", string_sec, column_sec
      FROM new_times
      WHERE string_sec != column_sec;   -- Only show rows where there IS a difference/discrepancy
    """).df()

if df.shape[0] == 0:
    print("All rows match between these two columns \n")
else:
    print("Unmatching rows found! See table below: \n")
display(df)

All rows match between these two columns 



,Total duration,Total duration (ms),string_sec,column_sec


- All time durations match between these two columns, which means we can focus on the millisecond column for EDA of journey times and ignore the Total duration column.

In [18]:
#con.close() #Closing DB connection 

# Plan of Action

## Cleaning & Transforming

- There are only 185 rows of missing data which can be removed from the dataset **(see: 3.1. Check For Missing Data)**
- Maybe best to remove Rows with "End station" of 'Mechanical Workshop Penton' and 'London Fields, Hackney'. They do not seem to be geniune cycle hires. **(See: 3.3.8. Checking: "Start station" and "Start station number" Match with "End station" and "End station number")**
- Need to remove leading zeros from station numbers - this will remove duplicate station numbers for each station name. 
- Need to have one consistent TIMESTAMP (YYYY-MM-DD HH:MM) for start and end times of journeys
- Convert **Total duration** from VARCHAR to INTERVAL
- Can create new database table with daily weather information for 2024 and 2025 - Use SQL joins for deeper analysis
- Could also obtain GPS coordinates of stations to estimate distance travelled per cycle journey 